In [3]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

load_dotenv()

True

In [4]:
llm = ChatOpenAI(model='gpt-4o-mini')

In [7]:
prompt = ChatPromptTemplate.from_template(
    'Reply in one sentence: {question}'
)

parser = StrOutputParser()
chain = prompt | llm 

In [9]:
chain.invoke('hi').content

'Hello! How can I assist you today?'

In [10]:
classify_prompt = ChatPromptTemplate.from_template(
    'Classify this email in ONE word (Billing/Technical/Spam/Other):\n'
    'Subject: {subject}\nBody: {body}'
)

clasify_chain = classify_prompt | llm | parser

In [11]:
emails = [
    {'subject': 'Payment failed',   'body': 'Charged twice this month.'},
    {'subject': 'App crash',        'body': 'Crashes on Settings page.'},
    {'subject': 'Add Slack?',       'body': 'Want Slack notifications.'},
    {'subject': 'WIN FREE IPHONE',  'body': 'Click here NOW!!!'},
    {'subject': 'Invoice question', 'body': 'Can I get annual invoice?'},
]

batch_results = clasify_chain.batch(emails,
                                    config={'max_concurrency':3})

In [12]:
batch_results

['Billing', 'Technical', 'Other', 'Spam', 'Billing']

In [13]:
# lets suppose you have your chain -->

outputs_from_llm = [
    'CATEGORY: Billing\nURGENCY: High',        # normal
    'CATEGORY:  Billing\nURGENCY: High',       # extra space!
    'Category: Billing\nUrgency: High',         # different case!
    'CATEGORY: Billing (payment issue)\nURGENCY: High',  # extra text!
    'I think this is CATEGORY: Billing\nURGENCY: High',  # explanation first!
]

print('Parsing with split() — watch it break:')
print('─' * 60)

for output in outputs_from_llm:
    category = 'PARSE FAILED'
    for line in output.split('\n'):
        if 'CATEGORY:' in line:  # exact match — brittle!
            category = line.replace('CATEGORY:', '').strip()
            break
    print(f'Input: {output[:40]:40} → Category: "{category}"')

Parsing with split() — watch it break:
────────────────────────────────────────────────────────────
Input: CATEGORY: Billing
URGENCY: High          → Category: "Billing"
Input: CATEGORY:  Billing
URGENCY: High         → Category: "Billing"
Input: Category: Billing
Urgency: High          → Category: "PARSE FAILED"
Input: CATEGORY: Billing (payment issue)
URGENC → Category: "Billing (payment issue)"
Input: I think this is CATEGORY: Billing
URGENC → Category: "I think this is  Billing"


In [14]:
# pydantic base model ( baemodel)

# let me define my output sturucture

from pydantic import BaseModel, Field
from typing import Literal, List

class EmailClassification(BaseModel):
    category : Literal[
        'Billing',
        'Technical', 
        'Feature Request',
        'Churn Risk',
        'Spam',
        'Other'
    ] = Field(description="The category of the support email")

    urgency: Literal['High', 'Medium', 'Low'] = Field(
        description='Urgency based on business impact'
    )
    confidence: int = Field(
        description='Confidence score from 1 to 10',
        ge=1,   # ge = greater than or equal (Pydantic v2!)
        le=10   # le = less than or equal
    )
    reasoning: str = Field(
        description='One sentence explanation of the classification'
    )

    cot_steps: List[str] = Field(
        description='3-5 Chain of Thought reasoning steps'
    )




In [ ]:
EmailClassification(
     category='Technical',
    urgency='High',
    confidence='23',
    reasoning='Team cannot login after paying → Billing issue',
    cot_steps=['Email mentions payment', 'Login is blocked', 'Category = Billing']
)

ValidationError: 1 validation error for EmailClassification
confidence
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='A', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/int_parsing